# Model Prototyping
Visualize reconstruction quality and reconstruction error distributions after training.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

from src.models.lstm_autoencoder import GRUAutoencoder
from src.models.threshold import load_threshold
from src.data.preprocessor import load_scaler
from src.data.ciciot_loader import load_dataset, make_windows

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

DATASET = 'ciciot'
dcfg = cfg['data'][DATASET]
mcfg = cfg['model']

In [ ]:
_, _, X_test_raw, y_test = load_dataset(
    csv_path='../'+dcfg['raw_path'],
    scaler_path='../'+dcfg['scaler_path'],
)
scaler = load_scaler('../'+dcfg['scaler_path'])
X_test_scaled = scaler.transform(X_test_raw)
X_windows = make_windows(X_test_scaled, mcfg['window_size'])
y_windows = y_test[mcfg['window_size']-1 : mcfg['window_size']-1+len(X_windows)]

In [ ]:
model = GRUAutoencoder(
    input_size=X_windows.shape[2],
    hidden_size=mcfg['hidden_size'],
    bottleneck_size=mcfg['bottleneck_size'],
    seq_len=mcfg['window_size'],
)
model.load_state_dict(torch.load('../'+cfg['saved_models'][f'{DATASET}_model'], map_location='cpu'))
model.eval()
threshold = load_threshold('../'+cfg['saved_models'][f'{DATASET}_threshold'])

In [ ]:
# Reconstruction error distribution
with torch.no_grad():
    x_t = torch.tensor(X_windows[:2000], dtype=torch.float32)
    errors = model.reconstruction_error(x_t).numpy()

plt.figure(figsize=(10, 4))
plt.hist(errors[y_windows[:2000]==0], bins=100, alpha=0.6, label='Normal')
plt.hist(errors[y_windows[:2000]==1], bins=100, alpha=0.6, label='Attack')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold={threshold:.4f}')
plt.xlabel('Reconstruction Error')
plt.legend()
plt.title('Reconstruction Error: Normal vs Attack')
plt.show()

In [ ]:
# Signal reconstruction visualization
idx = 0
x_sample = torch.tensor(X_windows[idx:idx+1], dtype=torch.float32)
with torch.no_grad():
    x_hat = model(x_sample).numpy()[0]
x_orig = X_windows[idx]

fig, axes = plt.subplots(1, X_windows.shape[2], figsize=(16, 3))
from src.data.stream_simulator import CICIOT_FEATURES
for i, (ax, feat) in enumerate(zip(axes, CICIOT_FEATURES)):
    ax.plot(x_orig[:, i], label='Original')
    ax.plot(x_hat[:, i], label='Reconstructed', linestyle='--')
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle('Original vs Reconstructed Signal (Normal sample)')
plt.tight_layout()
plt.show()